# 🏗️ Pipeline de Transformación de Datos

Este notebook integra las transformaciones de:

- Data Cleaning
- Feature Engineering

El objetivo es construir un flujo reproducible que pueda aplicarse a cualquier dataset de entrada.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import joblib

import ipywidgets as widgets
from IPython.display import display

In [2]:
# ============================================================
# 📂 CARGA DE DATOS (VERSIÓN ROBUSTA - PIPELINE)
# ============================================================

import io
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# ============================================================
# 🧩 WIDGETS
# ============================================================

uploader = widgets.FileUpload(accept='.csv', multiple=False)
btn_upload = widgets.Button(description="📂 Cargar archivo", button_style='success')

ruta = widgets.Text(
    description="Ruta CSV:",
    placeholder="Ej: data/raw/archivo.csv"
)
btn_ruta = widgets.Button(description="Cargar desde ruta", button_style='info')

output_loader = widgets.Output()

# ============================================================
# 🔧 FUNCIÓN ROBUSTA DE LECTURA
# ============================================================

def leer_csv_seguro(path_or_buffer):
    try:
        return pd.read_csv(path_or_buffer)
    except:
        try:
            return pd.read_csv(path_or_buffer, sep=';')
        except:
            return pd.read_csv(path_or_buffer, encoding='latin1')

# ============================================================
# 📤 CARGA DESDE UPLOAD
# ============================================================

def cargar_upload(b):
    global df
    with output_loader:
        output_loader.clear_output()

        if not uploader.value:
            print("⚠️ Sube un archivo primero")
            return

        try:
            archivo = next(iter(uploader.value.values()))
            contenido = archivo['content']
            df = leer_csv_seguro(io.BytesIO(contenido))

            print("✅ Dataset cargado desde upload")
            print("📐 Dimensión:", df.shape)
            display(df.head())

        except Exception as e:
            print("❌ Error al cargar archivo:")
            print(e)

# ============================================================
# 📁 CARGA DESDE RUTA
# ============================================================

def cargar_ruta(b):
    global df
    with output_loader:
        output_loader.clear_output()

        ruta_input = ruta.value.strip()

        if not ruta_input:
            print("⚠️ Ingresa una ruta")
            return

        # Validar existencia
        if not os.path.exists(ruta_input):
            print("❌ La ruta no existe")
            print("👉 Verifica:", ruta_input)
            return

        try:
            df = leer_csv_seguro(ruta_input)

            print("✅ Dataset cargado desde ruta")
            print("📐 Dimensión:", df.shape)
            display(df.head())

        except Exception as e:
            print("❌ Error al leer el CSV:")
            print(e)

# ============================================================
# 🔗 EVENTOS
# ============================================================

btn_upload.on_click(cargar_upload)
btn_ruta.on_click(cargar_ruta)

# ============================================================
# 🖥️ UI FINAL
# ============================================================

display(widgets.VBox([
    widgets.HTML("<h3>📂 Carga de datos (Pipeline)</h3>"),

    widgets.HTML("<b>📤 Subir archivo</b>"),
    uploader,
    btn_upload,

    widgets.HTML("<b>📁 O cargar desde ruta</b>"),
    ruta,
    btn_ruta,

    output_loader
]))

## ⚙️ Paso 1: Configuración de limpieza

Selecciona qué transformaciones deseas aplicar al dataset.

In [ ]:
# ============================================================
# 🧹 DATA CLEANING PRO FINAL (CONSISTENTE + UX PRO)
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from scipy.stats import zscore

# ============================================================
# VARIABLES
# ============================================================

df_clean = None
history_df = pd.DataFrame(columns=["Variable","Acción","Detalle"])
snapshots = []

treated = {
    "nulls": set(),
    "outliers": set(),
    "types": set(),
    "encoding": set(),
    "scaling": set()
}

output_cleaning = widgets.Output()
output_history = widgets.Output()

# dropdowns
col_nulls = widgets.Dropdown(description="Col:")
col_outliers = widgets.Dropdown(description="Col:")
col_types = widgets.Dropdown(description="Col:")
col_encoding = widgets.Dropdown(description="Col:")
col_scale = widgets.Dropdown(description="Col:")

# ============================================================
# 🔍 DETECCIÓN POR SECCIÓN (FIX)
# ============================================================

def analizar_columnas(df, section):
    info = {}
    
    for col in df.columns:
        
        # 🔵 ya tratado
        if col in treated[section]:
            info[col] = "🔵"
            continue
        
        # NULOS
        if section == "nulls":
            if df[col].isnull().sum() > 0:
                info[col] = "🔴"
            else:
                info[col] = "🟢"
        
        # OUTLIERS
        elif section == "outliers":
            if pd.api.types.is_numeric_dtype(df[col]):
                try:
                    outliers = np.sum(np.abs(zscore(df[col].dropna())) > 3)
                except:
                    outliers = 0
                
                if outliers > 0:
                    info[col] = "🟠"
                else:
                    info[col] = "🟢"
            else:
                info[col] = "🟢"
        
        # TIPOS
        elif section == "types":
            if df[col].dtype == "object":
                info[col] = "🔴"
            else:
                info[col] = "🟢"
        
        # ENCODING
        elif section == "encoding":
            if df[col].dtype == "object" or str(df[col].dtype) == "category":
                info[col] = "🔴"
            else:
                info[col] = "🟢"
        
        # ESCALADO
        elif section == "scaling":
            if pd.api.types.is_numeric_dtype(df[col]):
                info[col] = "🔴"
            else:
                info[col] = "🟢"
    
    return info

# ============================================================
# INIT
# ============================================================

btn_init = widgets.Button(description="Inicializar cleaning", button_style='primary')

def init_cleaning(b):
    global df_clean, snapshots
    
    with output_cleaning:
        output_cleaning.clear_output()
        
        if 'df' not in globals():
            print("❌ Primero carga datos")
            return
        
        df_clean = df.copy()
        snapshots = [df_clean.copy()]
        
        refresh_dropdowns()
        print("✅ Dataset listo")

btn_init.on_click(init_cleaning)

# ============================================================
# REFRESH
# ============================================================

def refresh_dropdowns():
    if df_clean is None:
        return
    
    col_nulls.options = [f"{analizar_columnas(df_clean,'nulls')[c]} {c}" for c in df_clean.columns]
    col_outliers.options = [f"{analizar_columnas(df_clean,'outliers')[c]} {c}" for c in df_clean.columns]
    col_types.options = [f"{analizar_columnas(df_clean,'types')[c]} {c}" for c in df_clean.columns]
    col_encoding.options = [f"{analizar_columnas(df_clean,'encoding')[c]} {c}" for c in df_clean.columns]
    col_scale.options = [f"{analizar_columnas(df_clean,'scaling')[c]} {c}" for c in df_clean.columns]

# ============================================================
# UTIL
# ============================================================

def get_col(drop):
    return drop.value.split(" ",1)[1]

def save():
    snapshots.append(df_clean.copy())

def log(v,a,d):
    global history_df
    history_df = pd.concat([history_df, pd.DataFrame({
        "Variable":[v],"Acción":[a],"Detalle":[d]
    })], ignore_index=True)
    actualizar_historial()

def actualizar_historial():
    with output_history:
        output_history.clear_output()
        display(history_df)

# ============================================================
# 🧠 NULOS
# ============================================================

legend_nulls = widgets.HTML("""
<b>🔵 NULOS</b><br>
🔴 Tiene nulos<br>
🔵 Ya imputado<br>
🟢 OK
""")

impute_method = widgets.Dropdown(options=['mean','median','mode'])
btn_impute = widgets.Button(description="Aplicar")

def imputar(b):
    global df_clean
    
    with output_cleaning:
        output_cleaning.clear_output()
        
        col = get_col(col_nulls)
        save()
        
        if impute_method.value == 'mean':
            df_clean[col].fillna(df_clean[col].mean(), inplace=True)
        elif impute_method.value == 'median':
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
        else:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
        
        treated["nulls"].add(col)
        log(col,"Imputación",impute_method.value)
        
        refresh_dropdowns()
        print("✅ Imputado")

btn_impute.on_click(imputar)

# ============================================================
# 📉 OUTLIERS
# ============================================================

legend_outliers = widgets.HTML("""
<b>🟠 OUTLIERS</b><br>
🟠 Tiene outliers<br>
🔵 Tratado<br>
🟢 OK
""")

outlier_method = widgets.Dropdown(options=['IQR','Z-score'])
param = widgets.FloatSlider(value=1.5,min=1,max=3)

btn_outliers = widgets.Button(description="Aplicar")

def tratar_outliers(b):
    global df_clean
    
    with output_cleaning:
        output_cleaning.clear_output()
        
        col = get_col(col_outliers)
        save()
        
        if outlier_method.value == 'IQR':
            Q1,Q3 = df_clean[col].quantile([0.25,0.75])
            IQR = Q3-Q1
            df_clean = df_clean[
                (df_clean[col]>=Q1-param.value*IQR)&
                (df_clean[col]<=Q3+param.value*IQR)
            ]
        else:
            df_clean = df_clean[np.abs(zscore(df_clean[col]))<param.value]
        
        treated["outliers"].add(col)
        log(col,"Outliers",outlier_method.value)
        
        refresh_dropdowns()
        print("✅ Outliers")

btn_outliers.on_click(tratar_outliers)

# ============================================================
# 🧩 TIPOS
# ============================================================

legend_types = widgets.HTML("""
<b>🟣 TIPOS</b><br>
🔴 Incorrecto<br>
🔵 Corregido<br>
🟢 OK
""")

type_method = widgets.Dropdown(options=['datetime','category','numeric'])
btn_types = widgets.Button(description="Aplicar")

def tipos(b):
    global df_clean
    
    with output_cleaning:
        output_cleaning.clear_output()
        
        col = get_col(col_types)
        save()
        
        if type_method.value=='datetime':
            df_clean[col]=pd.to_datetime(df_clean[col],errors='coerce')
        elif type_method.value=='category':
            df_clean[col]=df_clean[col].astype('category')
        else:
            df_clean[col]=pd.to_numeric(df_clean[col],errors='coerce')
        
        treated["types"].add(col)
        log(col,"Tipo",type_method.value)
        
        refresh_dropdowns()
        print("✅ Tipo corregido")

btn_types.on_click(tipos)

# ============================================================
# 🔠 ENCODING
# ============================================================

legend_encoding = widgets.HTML("""
<b>🟡 ENCODING</b><br>
🔴 Sin encoding<br>
🔵 Aplicado<br>
🟢 OK
""")

encoding_method = widgets.Dropdown(options=['OneHot','Ordinal'])
btn_encoding = widgets.Button(description="Aplicar")

def encoding(b):
    global df_clean
    
    with output_cleaning:
        output_cleaning.clear_output()
        
        col = get_col(col_encoding)
        save()
        
        if encoding_method.value=='OneHot':
            df_clean = pd.get_dummies(df_clean, columns=[col])
        else:
            df_clean[col]=df_clean[col].astype('category').cat.codes()
        
        treated["encoding"].add(col)
        log(col,"Encoding",encoding_method.value)
        
        refresh_dropdowns()
        print("✅ Encoding")

btn_encoding.on_click(encoding)

# ============================================================
# 📊 ESCALADO
# ============================================================

legend_scaling = widgets.HTML("""
<b>🟢 ESCALADO</b><br>
🔴 Sin escalar<br>
🔵 Aplicado<br>
🟢 OK
""")

scaler_method = widgets.Dropdown(options=['Standard','MinMax','Robust'])
btn_scale = widgets.Button(description="Aplicar")

def escalar(b):
    global df_clean
    
    with output_cleaning:
        output_cleaning.clear_output()
        
        col = get_col(col_scale)
        save()
        
        if scaler_method.value=='Standard':
            scaler=StandardScaler()
        elif scaler_method.value=='MinMax':
            scaler=MinMaxScaler()
        else:
            scaler=RobustScaler()
        
        df_clean[[col]]=scaler.fit_transform(df_clean[[col]])
        
        treated["scaling"].add(col)
        log(col,"Scaling",scaler_method.value)
        
        refresh_dropdowns()
        print("✅ Escalado")

btn_scale.on_click(escalar)

# ============================================================
# UI
# ============================================================

display(widgets.VBox([

    widgets.HTML("<h2>🧹 Data Cleaning PRO FINAL</h2>"),
    
    btn_init,
    
    widgets.HTML("<hr><h3>1️⃣ Nulos</h3>"),
    legend_nulls,
    col_nulls, impute_method, btn_impute,
    
    widgets.HTML("<hr><h3>2️⃣ Outliers</h3>"),
    legend_outliers,
    col_outliers, outlier_method, param, btn_outliers,
    
    widgets.HTML("<hr><h3>3️⃣ Tipos</h3>"),
    legend_types,
    col_types, type_method, btn_types,
    
    widgets.HTML("<hr><h3>4️⃣ Encoding</h3>"),
    legend_encoding,
    col_encoding, encoding_method, btn_encoding,
    
    widgets.HTML("<hr><h3>5️⃣ Escalado</h3>"),
    legend_scaling,
    col_scale, scaler_method, btn_scale,
    
    widgets.HTML("<hr><h3>Historial</h3>"),
    output_history,
    
    output_cleaning
]))

## 🧠 Paso 2: Feature Engineering

Selecciona las transformaciones para generar nuevas variables.

In [12]:
# ============================================================
# 🧠 FEATURE ENGINEERING PRO FINAL (COMPLETO + UX + COLORES)
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import PowerTransformer
from scipy.stats import zscore

import ipywidgets as widgets
from IPython.display import display

# ============================================================
# VARIABLES
# ============================================================

df_feat = None
history_feat = pd.DataFrame(columns=["Feature","Acción","Detalle"])
snapshots_feat = []

output_feat = widgets.Output()
output_feat_history = widgets.Output()

treated_feat = {
    "ratio": set(),
    "diff": set(),
    "agg": set(),
    "fecha": set(),
    "binning": set(),
    "interaction": set(),
    "transform": set(),
    "selection": set()
}

# ============================================================
# 🔍 DETECCIÓN VISUAL POR SECCIÓN
# ============================================================

def analizar_feat(df, section):
    info = {}

    for col in df.columns:

        if col in treated_feat[section]:
            info[col] = "🔵"
            continue

        if section in ["ratio","diff","interaction"]:
            info[col] = "🔴" if pd.api.types.is_numeric_dtype(df[col]) else "🟢"

        elif section == "agg":
            info[col] = "🔴"

        elif section == "fecha":
            info[col] = "🔴" if "date" in col.lower() or "fecha" in col.lower() else "🟢"

        elif section in ["binning","transform","selection"]:
            info[col] = "🔴" if pd.api.types.is_numeric_dtype(df[col]) else "🟢"

    return info

# ============================================================
# DROPDOWNS
# ============================================================

def make_dropdown():
    return widgets.Dropdown(description="Col:")

col_ratio_1 = make_dropdown()
col_ratio_2 = make_dropdown()

col_diff_1 = make_dropdown()
col_diff_2 = make_dropdown()

col_agg = make_dropdown()
agg_func = widgets.Dropdown(options=['mean','sum','max','min'])

col_fecha = make_dropdown()
col_bin = make_dropdown()

col_inter_1 = make_dropdown()
col_inter_2 = make_dropdown()

col_transform = make_dropdown()
col_select = make_dropdown()

# ============================================================
# REFRESH
# ============================================================

def refresh_feat():
    if df_feat is None:
        return

    def build(section):
        info = analizar_feat(df_feat, section)
        return [f"{info[c]} {c}" for c in df_feat.columns]

    col_ratio_1.options = build("ratio")
    col_ratio_2.options = build("ratio")

    col_diff_1.options = build("diff")
    col_diff_2.options = build("diff")

    col_agg.options = build("agg")
    col_fecha.options = build("fecha")

    col_bin.options = build("binning")

    col_inter_1.options = build("interaction")
    col_inter_2.options = build("interaction")

    col_transform.options = build("transform")
    col_select.options = build("selection")

# ============================================================
# UTIL
# ============================================================

def get_col(drop):
    return drop.value.split(" ",1)[1]

def save_feat():
    snapshots_feat.append(df_feat.copy())

def log_feat(f,a,d):
    global history_feat
    history_feat = pd.concat([history_feat, pd.DataFrame({
        "Feature":[f],"Acción":[a],"Detalle":[d]
    })], ignore_index=True)
    update_hist_feat()

def update_hist_feat():
    with output_feat_history:
        output_feat_history.clear_output()
        display(history_feat)

# ============================================================
# INIT
# ============================================================

btn_init = widgets.Button(description="Inicializar", button_style='primary')

def init_feat(b):
    global df_feat, snapshots_feat

    with output_feat:
        output_feat.clear_output()

        if 'df_clean' not in globals():
            print("❌ Ejecuta cleaning primero")
            return

        df_feat = df_clean.copy()
        snapshots_feat = [df_feat.copy()]

        refresh_feat()
        print("✅ Listo")

btn_init.on_click(init_feat)

# ============================================================
# 🧮 RATIO
# ============================================================

legend_ratio = widgets.HTML("🔴 numéricas | 🔵 usado | 🟢 no aplica")

def ratio(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        c1,c2 = get_col(col_ratio_1), get_col(col_ratio_2)

        if (c1,c2) in treated_feat["ratio"]:
            print("⚠️ Ya aplicado")
            return

        save_feat()
        new = f"{c1}_div_{c2}"
        df_feat[new] = df_feat[c1]/(df_feat[c2]+1e-5)

        treated_feat["ratio"].add((c1,c2))
        log_feat(new,"Ratio",f"{c1}/{c2}")

        refresh_feat()
        print("✅ Ratio")

btn_ratio = widgets.Button(description="Aplicar")
btn_ratio.on_click(ratio)

# ============================================================
# ➖ DIFERENCIAS
# ============================================================

legend_diff = widgets.HTML("🔴 numéricas | 🔵 usado")

def diff(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        c1,c2 = get_col(col_diff_1), get_col(col_diff_2)

        save_feat()
        new = f"{c1}_minus_{c2}"
        df_feat[new] = df_feat[c1]-df_feat[c2]

        treated_feat["diff"].add((c1,c2))
        log_feat(new,"Diferencia",f"{c1}-{c2}")

        refresh_feat()
        print("✅ Diferencia")

btn_diff = widgets.Button(description="Aplicar")
btn_diff.on_click(diff)

# ============================================================
# 📊 AGREGACIONES
# ============================================================

legend_agg = widgets.HTML("🔴 todas | 🔵 usado")

def agg(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        col = get_col(col_agg)

        save_feat()
        new = f"{col}_{agg_func.value}"
        df_feat[new] = getattr(df_feat[col], agg_func.value)()

        treated_feat["agg"].add(col)
        log_feat(new,"Agregación",agg_func.value)

        refresh_feat()
        print("✅ Agregado")

btn_agg = widgets.Button(description="Aplicar")
btn_agg.on_click(agg)

# ============================================================
# 📅 FECHAS
# ============================================================

legend_fecha = widgets.HTML("🔴 fechas | 🔵 usado")

def fecha(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        col = get_col(col_fecha)

        save_feat()
        df_feat[col] = pd.to_datetime(df_feat[col],errors='coerce')

        df_feat[f"{col}_year"] = df_feat[col].dt.year
        df_feat[f"{col}_month"] = df_feat[col].dt.month
        df_feat[f"{col}_day"] = df_feat[col].dt.day
        df_feat[f"{col}_dow"] = df_feat[col].dt.dayofweek

        treated_feat["fecha"].add(col)
        log_feat(col,"Fecha","year/month/day/dow")

        refresh_feat()
        print("✅ Fecha")

btn_fecha = widgets.Button(description="Aplicar")
btn_fecha.on_click(fecha)

# ============================================================
# 📊 BINNING
# ============================================================

legend_bin = widgets.HTML("🔴 numéricas | 🔵 usado")

bins = widgets.IntSlider(value=4,min=2,max=10)

def binning(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        col = get_col(col_bin)

        save_feat()
        df_feat[f"{col}_bin"] = pd.cut(df_feat[col],bins=bins.value)

        treated_feat["binning"].add(col)
        log_feat(col,"Binning",bins.value)

        refresh_feat()
        print("✅ Binning")

btn_bin = widgets.Button(description="Aplicar")
btn_bin.on_click(binning)

# ============================================================
# 🔗 INTERACCIONES
# ============================================================

legend_inter = widgets.HTML("🔴 numéricas | 🔵 usado")

def inter(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        c1,c2 = get_col(col_inter_1), get_col(col_inter_2)

        save_feat()
        new = f"{c1}_x_{c2}"
        df_feat[new] = df_feat[c1]*df_feat[c2]

        treated_feat["interaction"].add((c1,c2))
        log_feat(new,"Interacción",f"{c1}*{c2}")

        refresh_feat()
        print("✅ Interacción")

btn_inter = widgets.Button(description="Aplicar")
btn_inter.on_click(inter)

# ============================================================
# 🔄 TRANSFORMACIONES
# ============================================================

legend_transform = widgets.HTML("🔴 numéricas | 🔵 usado")

transform_method = widgets.Dropdown(options=['log','yeo-johnson'])

def transform(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        col = get_col(col_transform)

        save_feat()

        if transform_method.value == "log":
            df_feat[col] = np.log1p(df_feat[col])
        else:
            pt = PowerTransformer()
            df_feat[[col]] = pt.fit_transform(df_feat[[col]])

        treated_feat["transform"].add(col)
        log_feat(col,"Transform",transform_method.value)

        refresh_feat()
        print("✅ Transform")

btn_transform = widgets.Button(description="Aplicar")
btn_transform.on_click(transform)

# ============================================================
# 📉 SELECCIÓN
# ============================================================

legend_select = widgets.HTML("🔴 numéricas | 🔵 aplicado")

corr = widgets.FloatSlider(value=0.9,min=0.7,max=1)

def select(b):
    global df_feat
    with output_feat:
        output_feat.clear_output()

        save_feat()

        corr_m = df_feat.corr(numeric_only=True).abs()
        upper = corr_m.where(np.triu(np.ones(corr_m.shape),k=1).astype(bool))
        drop = [c for c in upper.columns if any(upper[c]>corr.value)]

        df_feat.drop(columns=drop,inplace=True)

        log_feat("multiple","Selección",f"corr>{corr.value}")

        refresh_feat()
        print("✅ Selección")

btn_select = widgets.Button(description="Aplicar")
btn_select.on_click(select)

# ============================================================
# UI
# ============================================================

display(widgets.VBox([

    widgets.HTML("<h2>🧠 Feature Engineering PRO FINAL</h2>"),
    btn_init,

    widgets.HTML("<hr><h3>1️⃣ Ratio</h3>"),
    legend_ratio, col_ratio_1, col_ratio_2, btn_ratio,

    widgets.HTML("<hr><h3>2️⃣ Diferencias</h3>"),
    legend_diff, col_diff_1, col_diff_2, btn_diff,

    widgets.HTML("<hr><h3>3️⃣ Agregaciones</h3>"),
    legend_agg, col_agg, agg_func, btn_agg,

    widgets.HTML("<hr><h3>4️⃣ Fechas</h3>"),
    legend_fecha, col_fecha, btn_fecha,

    widgets.HTML("<hr><h3>5️⃣ Binning</h3>"),
    legend_bin, col_bin, bins, btn_bin,

    widgets.HTML("<hr><h3>6️⃣ Interacciones</h3>"),
    legend_inter, col_inter_1, col_inter_2, btn_inter,

    widgets.HTML("<hr><h3>7️⃣ Transformación</h3>"),
    legend_transform, col_transform, transform_method, btn_transform,

    widgets.HTML("<hr><h3>8️⃣ Selección</h3>"),
    legend_select, corr, btn_select,

    widgets.HTML("<hr><h3>Historial</h3>"),
    output_feat_history,

    output_feat
]))

## 🏗️ Paso 3: Construcción del Pipeline

Se construye un pipeline reproducible de transformación.

In [ ]:
# ============================================================
# ⚖️ CLASS BALANCER PRO UX (COLORES + GUIADO + SIN BUGS)
# ============================================================

import pandas as pd
from sklearn.model_selection import train_test_split

from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.combine import SMOTEENN, SMOTETomek

import ipywidgets as widgets
from IPython.display import display

# ============================================================
# VARIABLES
# ============================================================

df_bal = None

history_bal = pd.DataFrame(columns=["Acción","Detalle"])
snapshots_bal = []

output_balance = widgets.Output()
output_balance_history = widgets.Output()

target_selector = widgets.Dropdown(description="Target:")

treated_bal = {
    "check": False,
    "split": False,
    "balance": False
}

# ============================================================
# 🔍 DETECCIÓN VISUAL TARGET
# ============================================================

def analizar_target(df):
    info = {}
    
    for col in df.columns:
        if col == target_selector.value and treated_bal["balance"]:
            info[col] = "🔵"
        else:
            info[col] = "🔴"
    
    return info

# ============================================================
# INIT (USANDO FEATURE ENGINEERING)
# ============================================================

btn_init = widgets.Button(description="Inicializar", button_style='primary')

def init_bal(b):
    global df_bal
    
    with output_balance:
        output_balance.clear_output()
        
        if 'df_feat' not in globals():
            print("❌ Ejecuta feature engineering primero")
            return
        
        df_bal = df_feat.copy()
        
        target_selector.options = df_bal.columns
        target_selector.value = df_bal.columns[-1]
        
        print("✅ Dataset listo para balanceo")

btn_init.on_click(init_bal)

# ============================================================
# 📊 VERIFICAR BALANCE
# ============================================================

legend_check = widgets.HTML("""
<b>📊 Balance</b><br>
🔴 revisar distribución<br>
🔵 ya analizado
""")

btn_check = widgets.Button(description="Ver distribución")

def check_balance(b):
    with output_balance:
        output_balance.clear_output()
        
        target = target_selector.value
        
        dist = df_bal[target].value_counts(normalize=True)
        display(dist)
        
        min_class = dist.min()
        
        if min_class < 0.3:
            print("⚠️ Desbalanceado (<30%) → balancear")
        else:
            print("✅ Balanceado")
        
        treated_bal["check"] = True

btn_check.on_click(check_balance)

# ============================================================
# ✂️ SPLIT
# ============================================================

legend_split = widgets.HTML("""
<b>✂️ Split</b><br>
🔴 no aplicado<br>
🔵 aplicado<br>
IMPORTANTE: evita data leakage
""")

test_size_slider = widgets.FloatSlider(value=0.2,min=0.1,max=0.4)

btn_split = widgets.Button(description="Aplicar split")

def split(b):
    global X_train, X_test, y_train, y_test
    
    with output_balance:
        output_balance.clear_output()
        
        target = target_selector.value
        
        X = df_bal.drop(columns=[target])
        y = df_bal[target]
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size_slider.value,
            stratify=y,
            random_state=42
        )
        
        treated_bal["split"] = True
        
        print("✅ Split correcto (solo train se balancea)")

btn_split.on_click(split)

# ============================================================
# ⚙️ MÉTODOS
# ============================================================

legend_method = widgets.HTML("""
<b>⚙️ Métodos</b><br>

🔵 Oversampling (recomendado)<br>
SMOTE, ADASYN<br><br>

🟠 Undersampling (pierde info)<br>
NearMiss<br><br>

🟣 Híbridos (mejor balance)<br>
SMOTEENN, SMOTETomek
""")

method_selector = widgets.Dropdown(options=[
    'RandomOverSampler',
    'SMOTE',
    'ADASYN',
    'RandomUnderSampler',
    'NearMiss',
    'SMOTEENN',
    'SMOTETomek'
])

# ============================================================
# ⚖️ BALANCEO
# ============================================================

legend_balance = widgets.HTML("""
<b>⚖️ Balanceo</b><br>
🔴 pendiente<br>
🔵 aplicado<br>
SOLO en TRAIN
""")

btn_balance = widgets.Button(description="Aplicar balanceo")

def balance(b):
    global X_train, y_train
    
    with output_balance:
        output_balance.clear_output()
        
        if not treated_bal["split"]:
            print("⚠️ Primero hacer split")
            return
        
        if treated_bal["balance"]:
            print("⚠️ Ya aplicado")
            return
        
        snapshots_bal.append((X_train.copy(), y_train.copy()))
        
        method = method_selector.value
        
        if method == 'RandomOverSampler':
            sampler = RandomOverSampler()
        elif method == 'SMOTE':
            sampler = SMOTE()
        elif method == 'ADASYN':
            sampler = ADASYN()
        elif method == 'RandomUnderSampler':
            sampler = RandomUnderSampler()
        elif method == 'NearMiss':
            sampler = NearMiss()
        elif method == 'SMOTEENN':
            sampler = SMOTEENN()
        else:
            sampler = SMOTETomek()
        
        X_train, y_train = sampler.fit_resample(X_train, y_train)
        
        treated_bal["balance"] = True
        
        dist = pd.Series(y_train).value_counts(normalize=True)
        display(dist)
        
        history_bal.loc[len(history_bal)] = ["Balanceo", method]
        actualizar_historial()
        
        print(f"✅ Balance aplicado: {method}")

btn_balance.on_click(balance)

# ============================================================
# 💾 GUARDAR
# ============================================================

btn_save = widgets.Button(description="Guardar")

def guardar(b):
    with output_balance:
        output_balance.clear_output()
        
        df_train = X_train.copy()
        df_train[target_selector.value] = y_train
        
        df_train.to_csv("data/processed/balanced_train.csv", index=False)
        
        print("💾 Guardado")

btn_save.on_click(guardar)

# ============================================================
# HISTORIAL
# ============================================================

def actualizar_historial():
    with output_balance_history:
        output_balance_history.clear_output()
        display(history_bal)

# ============================================================
# UI
# ============================================================

display(widgets.VBox([

    widgets.HTML("<h2>⚖️ Balanceo de clases PRO</h2>"),
    
    btn_init,
    
    widgets.HTML("<hr><h3>1️⃣ Verificar</h3>"),
    legend_check,
    target_selector,
    btn_check,
    
    widgets.HTML("<hr><h3>2️⃣ Split</h3>"),
    legend_split,
    test_size_slider,
    btn_split,
    
    widgets.HTML("<hr><h3>3️⃣ Método</h3>"),
    legend_method,
    method_selector,
    
    widgets.HTML("<hr><h3>4️⃣ Balancear</h3>"),
    legend_balance,
    btn_balance,
    
    widgets.HTML("<hr><h3>Historial</h3>"),
    output_balance_history,
    
    widgets.HTML("<hr><h3>Guardar</h3>"),
    btn_save,
    
    output_balance
]))